In [1]:
#Copyright 2026 Mücahit Sahin
#
#Licensed under the Apache License, Version 2.0 (the "License");
#you may not use this file except in compliance with the License.
#You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
#Unless required by applicable law or agreed to in writing, software
#distributed under the License is distributed on an "AS IS" BASIS,
#WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#See the License for the specific language governing permissions and
#limitations under the License.

In [2]:
import pandas as pd
import numpy as np
import os
import sys
import json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from utils import metrics
from utils import splitting

In [3]:
class_map = {
    0: 'numeric',
    1: 'categorical',
    2: 'datetime',
    3: 'sentence',
    4: 'url',
    5: 'embedded-number',
    6: 'list',
    7: 'not-generalizable',
    8: 'context-specific'
}

In [4]:
df = pd.read_csv('results/featurized_data_with_llm_features.csv')
df.head()

,total_vals,num_nans,%_nans,num_of_dist_val,%_dist_val,mean,std_dev,min_val,max_val,has_delimiters,...,Qwen_3_5_detect_all_at_once_fewshot,Qwen_3_5_is_numeric_fewshot,Qwen_3_5_is_categorical_fewshot,Qwen_3_5_is_datetime_fewshot,Qwen_3_5_is_sentence_fewshot,Qwen_3_5_is_url_fewshot,Qwen_3_5_is_embedded_number_fewshot,Qwen_3_5_is_list_fewshot,Qwen_3_5_is_not_generalizable_fewshot,Qwen_3_5_is_context_specific_fewshot
0,21477,0,0.0,174,0.810169,0.000000,0.000000,0.0,0.0,0,...,categorical,0,1,0,0,0,0,0,0,0
1,21477,0,0.0,174,0.810169,125.449411,72.866452,1.0,276.0,0,...,categorical,0,1,0,0,0,0,0,0,1
2,21477,0,0.0,2,0.009312,0.000000,0.000000,0.0,0.0,0,...,categorical,0,1,0,0,0,0,0,0,0
3,21477,0,0.0,2,0.009312,5211.687154,146.816661,5142.0,5521.0,0,...,not-generalizable,0,1,0,0,0,0,0,1,1
4,21477,0,0.0,115,0.535457,0.000000,0.000000,0.0,0.0,0,...,categorical,0,1,0,0,0,0,0,0,0


In [5]:
all_datasets = pd.read_csv('data/Benchmark-Labeled-Data/all_data.csv')

In [6]:
# evaluate raw LLM predictions on 100 resample splits

In [7]:
def resample_and_evaluate(df, pred_cols, n_splits, raw_data):
    # Create a list to store the results for each feature set on each test set
    results = {
        'numeric': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'categorical': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'datetime': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'sentence': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'url': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'embedded-number': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'list': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'not-generalizable': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'context-specific': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        },
        'direct': {
            'F1-Score': [],
            'Precision': [],
            'Recall': [],
            'Accuracy': [],
        }
    }

    # limit to 100 resamples at max
    if n_splits > 100:
        n_splits = 100
    
    with open('seeds.txt', 'r') as fp: # change the tolerance here
        seeds = fp.read().splitlines()
        seeds = [int(i) for i in seeds] # convert back to int
        fp.close()

    
    # Iterate over each feature set and its associated features    
    for col in pred_cols:
        # create the number of resample splits
        for i in range(n_splits):
            seed = seeds[i]
            outer_fold = splitting.create_split(raw_data, test_size=0.2, target_col='y_act', group_col='Record_id', random_state=seed)
            train = raw_data.iloc[outer_fold["train"]].reset_index(drop=True)
            train_featurized = df.iloc[outer_fold["train"]].reset_index(drop=True)
            test_featurized = df.iloc[outer_fold["test"]].reset_index(drop=True)
            
            inner_fold = splitting.create_split(train, test_size=0.25, target_col='y_act', group_col='Record_id', random_state=seed)
            inner_train_featurized = train_featurized.iloc[inner_fold["train"]].reset_index(drop=True)
            dev_featurized = train_featurized.iloc[inner_fold["test"]].reset_index(drop=True)
            

            if 'detect_all' in col:
                y_test = test_featurized["y_act"]
                y_pred = test_featurized[col]
                
                # Calculate performance
                acc_test = accuracy_score(y_test, y_pred)
                f1_test = f1_score(y_test, y_pred, average='macro')
                prec_test = precision_score(y_test, y_pred, average='macro')
                rec_test = recall_score(y_test, y_pred, average='macro')

                results['direct']['F1-Score'].append(f1_test)
                results['direct']['Precision'].append(prec_test)
                results['direct']['Recall'].append(rec_test)
                results['direct']['Accuracy'].append(acc_test)
                
            else:
                df2 = test_featurized.copy()
                c = ""
                if 'numeric' in col:
                    c = 'numeric'
                elif 'categorical' in col:
                    c = 'categorical'
                elif 'datetime' in col:
                    c = 'datetime'
                elif 'sentence' in col:
                    c = 'sentence'
                elif 'url' in col:
                    c = 'url'
                elif 'list' in col:
                    c = 'list'
                elif 'embedded' in col:
                    c = 'embedded-number'
                elif 'generalizable' in col:
                    c = 'not-generalizable'
                else:
                    c = 'context-specific'

                df2.loc[df2.loc[~(df2["y_act"]==c)].index, 'y_act'] = 0
                df2.loc[df2.loc[df2["y_act"]==c].index, 'y_act'] = 1
                
                y_test = list(df2["y_act"])
                y_pred = list(df2[col].astype('int'))
        
                # Calculate performance
                acc_test = accuracy_score(y_test, y_pred)
                f1_test = f1_score(y_test, y_pred, average='macro')
                prec_test = precision_score(y_test, y_pred, average='macro')
                rec_test = recall_score(y_test, y_pred, average='macro')

                results[c]['F1-Score'].append(f1_test)
                results[c]['Precision'].append(prec_test)
                results[c]['Recall'].append(rec_test)
                results[c]['Accuracy'].append(acc_test)

    macro_avg = {
        'F1-Score': [],
        'Precision': [],
        'Recall': [],
        'Accuracy': []
    }
    
    for r in results:
        for m in results[r]:
            if m == 'F1-Score' and not r == 'direct':
                macro_avg['F1-Score'].append(results[r][m])
            elif m == 'Precision' and not r == 'direct':
                macro_avg['Precision'].append(results[r][m])
            elif m == 'Recall' and not r == 'direct':
                macro_avg['Recall'].append(results[r][m])
            elif m == 'Accuracy' and not r == 'direct':
                macro_avg['Accuracy'].append(results[r][m])
            results[r][m] = round(np.mean(results[r][m]), 3)
                
    results['Macro Avg. F1-Score 9 Binary Classes'] = round(np.mean(macro_avg["F1-Score"]), 3)
    results['Macro Avg. Precision 9 Binary Classes'] = round(np.mean(macro_avg["Precision"]), 3)
    results['Macro Avg. Recall 9 Binary Classes'] = round(np.mean(macro_avg["Recall"]), 3)
    results['Macro Avg. Accuracy 9 Binary Classes'] = round(np.mean(macro_avg["Accuracy"]), 3)
    
    return results

In [8]:
n_splits = 100

In [9]:
df['y_act'] = all_datasets['y_act']

In [10]:
gpt_5_4_pred_columns = [c for c in df.columns[1478:] if ("GPT_5_4" in c and not "fewshot" in c)]

In [11]:
gpt_5_4 = resample_and_evaluate(df, gpt_5_4_pred_columns, n_splits, all_datasets)

In [12]:
gpt_5_4_fewshot_pred_columns = [c for c in df.columns[1478:] if ("GPT_5_4" in c and "fewshot" in c)]

In [13]:
gpt_5_4_fewshot = resample_and_evaluate(df, gpt_5_4_fewshot_pred_columns, n_splits, all_datasets)

In [14]:
gemma_4_pred_columns = [c for c in df.columns[1478:] if ("Gemma_4" in c and not "fewshot" in c)]

In [15]:
gemma_4 = resample_and_evaluate(df, gemma_4_pred_columns, n_splits, all_datasets)

In [16]:
gemma_4_fewshot_pred_columns = [c for c in df.columns[1478:] if ("Gemma_4" in c and "fewshot" in c)]

In [17]:
gemma_4_fewshot = resample_and_evaluate(df, gemma_4_fewshot_pred_columns, n_splits, all_datasets)

In [18]:
qwen_3_5_pred_columns = [c for c in df.columns[1478:] if ("Qwen_3_5" in c and not "fewshot" in c)]

In [19]:
qwen_3_5 = resample_and_evaluate(df, qwen_3_5_pred_columns, n_splits, all_datasets)

In [20]:
qwen_3_5_fewshot_pred_columns = [c for c in df.columns[1478:] if ("Qwen_3_5" in c and "fewshot" in c)]

In [21]:
qwen_3_5_fewshot = resample_and_evaluate(df, qwen_3_5_fewshot_pred_columns, n_splits, all_datasets)

# Macro Avg. F1-Score 9 Binary Classes

In [38]:
gpt_5_4['Macro Avg. F1-Score 9 Binary Classes']

0.901

In [39]:
gpt_5_4_fewshot['Macro Avg. F1-Score 9 Binary Classes']

0.912

In [44]:
gemma_4['Macro Avg. F1-Score 9 Binary Classes']

0.867

In [45]:
gemma_4_fewshot['Macro Avg. F1-Score 9 Binary Classes']

0.886

In [46]:
qwen_3_5['Macro Avg. F1-Score 9 Binary Classes']

0.868

In [47]:
qwen_3_5_fewshot['Macro Avg. F1-Score 9 Binary Classes']

0.894